# U2 · Dimensión Estaciones — 02. Streaming, inferencia en vivo y observabilidad

**Integrante:** Grimaldo Arredondo Martinez — Data Geniuses · **Proyecto Sello:** CitiBike Intelligent Mobility

Este notebook es la capa de velocidad de la arquitectura Lambda para la dimensión **Estaciones**. Corre tres consultas de Spark Structured Streaming en paralelo y un monitor:

| Consulta | Topic de entrada | Qué hace | Criterio U2 |
|---|---|---|---|
| `u2_demanda` | `citibike-rides` (eventos empresariales) | Salidas y llegadas por estación en ventanas de 15 min, con watermark y checkpoint; inferencia de la ventana t+1 con el modelo del notebook 01 | 1, 3, 6, 7 |
| `u2_estado_iot` | `citibike-estaciones` (telemetría GBFS de los docks) | Deduplica reportes repetidos del sensor y publica bicis/docks disponibles y ocupación | 2, 6 |
| `u2_calidad` | ambos | Validez del contrato, latencia de ingesta y bytes por evento, por topic | 4, 5 |
| monitor (hilo) | `lastProgress` de las 3 consultas | Throughput, duración de micro-lote, retraso de offsets, filas descartadas por watermark | 4 |

Todo se publica en Prometheus vía Pushgateway (patrón de S11) y se ve en el tablero **"U2 · Estaciones — Grimaldo"** de Grafana.

**Antes de correrlo:** Kafka, `obs/` y `u2-estaciones/` levantados, el notebook 01 ejecutado (genera el modelo y `estaciones_top.json`) y los dos productores corriendo (ver `u2-estaciones/README.md`).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("U2_Estaciones_02_Streaming_GrimaldoArredondo")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "America/New_York")   # igual que el notebook 01
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-34fc8db9-e16c-4db2-bf71-e5d7596b72cb;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central


	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 500ms :: artifacts dl 22ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collectio

26/10/30 03:07:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.2.0


## 1. Parámetros, modelo y artefactos del notebook 01

In [2]:
import json, os, shutil, threading, time, urllib.request
from collections import defaultdict, deque
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.ml import PipelineModel

KAFKA = "kafka:9092"
TOPIC_RIDES = "citibike-rides"
TOPIC_ESTACIONES = "citibike-estaciones"
PUSHGATEWAY = "http://pushgateway:9091"

ARTIFACTS = "./artifacts"
STREAM_OUT = f"{ARTIFACTS}/stream"            # Parquet de salida de las consultas
CHECKPOINTS = f"{ARTIFACTS}/checkpoints"
VENTANA_MIN = 15
WATERMARK_DEMANDA = "10 minutes"   # tiempo de evento (histórico): con SPEEDUP=60 equivale a ~10 s reales
WATERMARK_IOT = "15 minutes"       # sobre publishedAt: cuánto se recuerda un reporte para deduplicarlo
SPEEDUP = 60                       # el mismo valor que en u2-estaciones/compose.yml (para la sección de costos)
NY = ZoneInfo("America/New_York")

with open(f"{ARTIFACTS}/estaciones_top.json", encoding="utf-8") as f:
    TOP = json.load(f)["estaciones"]
IDS_TOP = [e["station_id"] for e in TOP]
NOMBRE = {e["station_id"]: e["station_name"] for e in TOP}

with open(f"{ARTIFACTS}/metricas_modelo.json", encoding="utf-8") as f:
    META = json.load(f)
FEATURES = META["features"]
assert META["ventana_min"] == VENTANA_MIN, "La ventana del modelo y la del streaming deben coincidir"

modelo = PipelineModel.load(f"{ARTIFACTS}/models/modelo_estaciones_ts")
MEDIA_SLOT = {(r["station_id"], r["finde"], r["hora"]): r["media_slot"]
              for r in spark.read.parquet(f"{ARTIFACTS}/media_slot").collect()}

print(f"Modelo: {META['ganador']} | {len(IDS_TOP)} estaciones top | {len(MEDIA_SLOT)} franjas de media histórica")

Modelo: LinearRegression | 20 estaciones top | 960 franjas de media histórica


## 2. Publicación en Pushgateway

Cada consulta publica en su propio grupo (`instance`): `demanda`, `iot`, `calidad` y `monitor`. Un `PUT` reemplaza el grupo completo. Así, si una estación sale del ranking de la ventana, su serie también desaparece del tablero en lugar de quedar con un valor congelado.

In [3]:
JOB = "lambda26_u2_estaciones"

def _esc(v):
    return str(v).replace("\\", "\\\\").replace('"', '\\"')

def metrica(_metrica, _valor, **labels):
    """Una línea en formato de exposición de Prometheus (los labels pueden llamarse como sea, incluso 'nombre')."""
    if _valor is None:
        return None
    lab = ",".join(f'{k}="{_esc(v)}"' for k, v in labels.items())
    return f"lambda26_u2est_{_metrica}{{{lab}}} {float(_valor)}" if lab else f"lambda26_u2est_{_metrica} {float(_valor)}"

def push(instancia, lineas):
    cuerpo = "\n".join(l for l in lineas if l) + "\n"
    req = urllib.request.Request(f"{PUSHGATEWAY}/metrics/job/{JOB}/instance/{instancia}",
                                 data=cuerpo.encode("utf-8"), method="PUT")
    try:
        urllib.request.urlopen(req, timeout=5).close()
    except Exception as ex:   # el streaming no debe caerse porque Grafana no esté levantado
        print(f"[push {instancia}] Pushgateway no disponible: {ex}")

## 3. Lectura de Kafka y contrato de eventos

El contrato completo está en `u2-estaciones/CONTRATO_U2.md`. Aquí se declara el mismo esquema y la regla `isValid`. Un evento que no cumple **no se descarta en silencio**: la consulta `u2_calidad` lo cuenta como inválido y el tablero lo muestra.

In [4]:
from pyspark.sql.types import StructType, StringType, LongType, IntegerType, BooleanType

def leer_topic(topics, starting="latest"):
    return (spark.readStream.format("kafka")
            .option("kafka.bootstrap.servers", KAFKA)
            .option("subscribe", topics)
            .option("startingOffsets", starting)
            .option("failOnDataLoss", "false")
            .option("maxOffsetsPerTrigger", 20000)      # techo por micro-lote (control de contrapresión)
            .load())

esquema_viaje = (StructType()
    .add("tipoEvento", StringType()).add("stationId", StringType()).add("stationName", StringType())
    .add("rideId", StringType()).add("rideableType", StringType()).add("memberCasual", StringType())
    .add("origen", StringType()).add("timestamp", LongType()).add("publishedAt", LongType()))

esquema_estado = (StructType()
    .add("tipoEvento", StringType()).add("stationId", StringType()).add("gbfsStationId", StringType())
    .add("stationName", StringType()).add("capacidad", IntegerType())
    .add("bicisDisponibles", IntegerType()).add("ebikesDisponibles", IntegerType())
    .add("docksDisponibles", IntegerType()).add("bicisInhabilitadas", IntegerType())
    .add("docksInhabilitados", IntegerType()).add("enServicio", BooleanType())
    .add("origen", StringType()).add("timestamp", LongType()).add("publishedAt", LongType()))

VALIDO_VIAJE = (F.col("tipoEvento").isin("viaje.salida", "viaje.llegada") & F.col("stationId").isNotNull()
                & (F.col("timestamp") > 0) & F.col("publishedAt").isNotNull())
VALIDO_ESTADO = ((F.col("tipoEvento") == "estacion.estado") & F.col("stationId").isNotNull()
                 & F.col("bicisDisponibles").between(0, 150) & F.col("docksDisponibles").between(0, 150)
                 & (F.col("timestamp") > 0) & F.col("publishedAt").isNotNull())

def parsear(raw, esquema, regla):
    return (raw.select(F.col("value").cast("string").alias("json"))
               .select(F.from_json("json", esquema).alias("e")).select("e.*")
               .withColumn("eventTime", F.timestamp_millis("timestamp"))
               .withColumn("publishedTime", F.timestamp_millis("publishedAt"))
               .withColumn("isValid", F.coalesce(regla, F.lit(False))))

viajes = parsear(leer_topic(TOPIC_RIDES), esquema_viaje, VALIDO_VIAJE)
estados = parsear(leer_topic(TOPIC_ESTACIONES), esquema_estado, VALIDO_ESTADO)

## 4. Consulta `u2_demanda`: ventanas + watermark + inferencia t+1

- **Ventana:** tumbling de 15 min sobre el **tiempo del evento** (la hora original del viaje), no sobre la hora de llegada a Kafka.
- **Watermark de 10 min:** una ventana se emite (modo `append`) cuando el watermark pasa su final. Un viaje que llega con más de 10 min de atraso respecto del evento más reciente se descarta, y Spark lo reporta en `numRowsDroppedByWatermark` (evidencia en la sección 8).
- **Checkpoint:** offsets y estado de las ventanas abiertas quedan en `artifacts/checkpoints/demanda`. Si la consulta se detiene, retoma donde quedó.

En `foreachBatch`, por cada ventana cerrada:
1. Se completa con **0** las estaciones top sin viajes (igual que la malla del notebook 01).
2. Se compara con la predicción hecha antes para esa ventana. Ese es el **MAE en producción**.
3. Se actualiza el historial de cada estación (`s_t … s_t3`) y se predice la ventana **t+1** con el modelo guardado.
4. Se publica en Grafana: observado, predicho para esta ventana, pronóstico de la siguiente, ranking top 10 y estaciones en alerta.

In [5]:
VENTANA_MS = VENTANA_MIN * 60 * 1000

def iniciar_estado():
    global historial, pendientes, eval_suma, eval_n, ultima_publicacion
    historial = defaultdict(lambda: deque(maxlen=4))   # salidas de las últimas 4 ventanas por estación
    pendientes = {}                                     # (estación, inicio_ventana_ms) -> predicción
    eval_suma, eval_n = 0.0, 0
    ultima_publicacion = {}

iniciar_estado()

def contexto_slot(inicio_ms):
    d = datetime.fromtimestamp(inicio_ms / 1000, NY)
    hora = d.hour + d.minute / 60.0
    finde = 1 if d.weekday() >= 5 else 0
    return hora, finde, d.hour

def features_siguiente(estacion, ventana_ms):
    """Features para predecir la ventana que empieza en ventana_ms + 15 min (misma definición que el notebook 01)."""
    obj = ventana_ms + VENTANA_MS
    hora, finde, h = contexto_slot(obj)
    media = MEDIA_SLOT.get((estacion, finde, h), 0.0)
    h_est = list(historial[estacion])                    # [más antigua ... más reciente]
    relleno = [media] * (4 - len(h_est))                 # arranque en frío: se completa con la media histórica
    s_t3, s_t2, s_t1, s_t = (relleno + h_est)[-4:]
    import math
    return {"station_id": estacion, "slot_obj_ms": obj,
            "s_t": float(s_t), "s_t1": float(s_t1), "s_t2": float(s_t2), "s_t3": float(s_t3),
            "l_t": float(ultima_publicacion.get(("l", estacion), 0)),
            "hora_sin": math.sin(2 * math.pi * hora / 24), "hora_cos": math.cos(2 * math.pi * hora / 24),
            "finde_obj": float(finde), "media_slot_obj": float(media)}

def procesar_ventanas(df, batch_id):
    global eval_suma, eval_n
    filas = (df.select(F.unix_millis("window.start").alias("w_ms"), "stationId", "nombre", "salidas", "llegadas")
               .collect())
    if not filas:
        return
    (df.select(F.lit(batch_id).alias("batchId"), F.col("window.start").alias("ventanaInicio"),
               F.col("window.end").alias("ventanaFin"), "stationId", "nombre", "salidas", "llegadas")
       .write.mode("append").parquet(f"{STREAM_OUT}/demanda_ventanas"))

    por_ventana = defaultdict(list)
    for r in filas:
        por_ventana[r["w_ms"]].append(r)

    evaluaciones, predicciones = [], []
    for w_ms in sorted(por_ventana):
        filas_w = por_ventana[w_ms]
        obs = {r["stationId"]: r for r in filas_w}
        # 1-2. completar con ceros y evaluar la predicción previa
        for est in IDS_TOP:
            s = obs[est]["salidas"] if est in obs else 0
            l = obs[est]["llegadas"] if est in obs else 0
            if (est, w_ms) in pendientes:
                pred = pendientes.pop((est, w_ms))
                evaluaciones.append((est, w_ms, s, pred, abs(pred - s)))
                eval_suma += abs(pred - s)
                eval_n += 1
                ultima_publicacion[("pred_para", est)] = pred
            historial[est].append(s)                      # 3. historial
            ultima_publicacion[("s", est)] = s
            ultima_publicacion[("l", est)] = l
        # 3. predecir t+1 para todas las estaciones top
        feats = [features_siguiente(est, w_ms) for est in IDS_TOP]
        pred_df = modelo.transform(spark.createDataFrame(feats)).select("station_id", "slot_obj_ms", "prediction").collect()
        for p in pred_df:
            valor = max(0.0, float(p["prediction"]))      # una demanda negativa no existe
            pendientes[(p["station_id"], p["slot_obj_ms"])] = valor
            ultima_publicacion[("pron", p["station_id"])] = valor
            ultima_publicacion[("media_sig", p["station_id"])] = feats[IDS_TOP.index(p["station_id"])]["media_slot_obj"]
            predicciones.append((batch_id, p["station_id"], w_ms, p["slot_obj_ms"], valor))
        ranking = sorted(filas_w, key=lambda r: -r["salidas"])[:10]
        ultima_publicacion["ranking"] = [(i + 1, r["stationId"], r["nombre"], r["salidas"]) for i, r in enumerate(ranking)]
        ultima_publicacion["total"] = sum(r["salidas"] for r in filas_w)
        ultima_publicacion["ventana_fin_ms"] = w_ms + VENTANA_MS

    if predicciones:
        spark.createDataFrame(predicciones, "batchId long, stationId string, ventanaBaseMs long, ventanaObjetivoMs long, prediccion double") \
             .write.mode("append").parquet(f"{STREAM_OUT}/predicciones")
    if evaluaciones:
        spark.createDataFrame(evaluaciones, "stationId string, ventanaMs long, real long, prediccion double, errorAbs double") \
             .write.mode("append").parquet(f"{STREAM_OUT}/evaluacion_online")

    # 4. Grafana
    L = []
    for est in IDS_TOP:
        lab = {"estacion": est, "nombre": NOMBRE[est]}
        pron, media = ultima_publicacion.get(("pron", est)), ultima_publicacion.get(("media_sig", est))
        L += [metrica("salidas_observadas", ultima_publicacion.get(("s", est)), **lab),
              metrica("llegadas_observadas", ultima_publicacion.get(("l", est)), **lab),
              metrica("salidas_predichas_para_ventana", ultima_publicacion.get(("pred_para", est)), **lab),
              metrica("salidas_pronostico_siguiente", pron, **lab),
              metrica("salidas_media_historica_siguiente", media, **lab)]
    alerta = [e for e in IDS_TOP if ultima_publicacion.get(("pron", e), 0) >= 1.5 * max(ultima_publicacion.get(("media_sig", e), 0), 1.0)]
    L += [metrica("ranking_salidas", s, puesto=f"{p:02d}", estacion=e, nombre=n) for p, e, n, s in ultima_publicacion["ranking"]]
    L += [metrica("salidas_total_ventana", ultima_publicacion["total"]),
          metrica("pronostico_total_siguiente_top", sum(ultima_publicacion.get(("pron", e), 0) for e in IDS_TOP)),
          metrica("estaciones_en_alerta_siguiente", len(alerta)),
          metrica("mae_online", eval_suma / eval_n if eval_n else None),
          metrica("predicciones_evaluadas", eval_n),
          metrica("ventana_fin_historica_ms", ultima_publicacion["ventana_fin_ms"]),
          metrica("ventana_actual_info", 1, ventana=datetime.fromtimestamp(
              (ultima_publicacion["ventana_fin_ms"] - VENTANA_MS) / 1000, NY).strftime("%Y-%m-%d %H:%M") + " NYC"),
          metrica("ultimo_batch", batch_id)]
    push("demanda", L)
    print(f"[demanda] batch {batch_id}: {len(por_ventana)} ventana(s) cerrada(s), "
          f"MAE online = {eval_suma / eval_n if eval_n else float('nan'):.3f} ({eval_n} evaluadas)")

demanda = (viajes.filter("isValid")
    .withWatermark("eventTime", WATERMARK_DEMANDA)
    .groupBy(F.window("eventTime", f"{VENTANA_MIN} minutes"), "stationId")
    .agg(F.sum(F.when(F.col("tipoEvento") == "viaje.salida", 1).otherwise(0)).alias("salidas"),
         F.sum(F.when(F.col("tipoEvento") == "viaje.llegada", 1).otherwise(0)).alias("llegadas"),
         F.first("stationName", ignorenulls=True).alias("nombre")))

## 5. Consulta `u2_estado_iot`: telemetría de los docks, deduplicada

El feed GBFS se actualiza cada pocos segundos, pero una estación tranquila puede repetir el mismo `last_reported` en varias lecturas seguidas. El productor publica todo lo que lee y **Spark deduplica** con `dropDuplicatesWithinWatermark(["stationId", "timestamp"])`. Así, un mismo reporte del sensor cuenta una sola vez. El watermark va sobre `publishedAt` (cuándo se leyó), no sobre `last_reported`. Si fuera sobre `last_reported`, una estación cuyo sensor lleva rato sin reportar se descartaría como dato tardío.

KPIs que se publican: bicis y e-bikes disponibles, docks libres, ocupación (%), estaciones **vacías** (0 bicis: nadie puede salir) y **llenas** (0 docks: nadie puede devolver), y la antigüedad del último reporte de cada sensor.

In [6]:
estado_actual = {}

def procesar_estado(df, batch_id):
    filas = df.select("stationId", "stationName", "capacidad", "bicisDisponibles", "ebikesDisponibles",
                      "docksDisponibles", "enServicio", "timestamp", "publishedAt").collect()
    if not filas:
        return
    df.withColumn("batchId", F.lit(batch_id)).write.mode("append").parquet(f"{STREAM_OUT}/estado_iot")
    for r in filas:                                    # se queda con el reporte más reciente de cada estación
        previo = estado_actual.get(r["stationId"])
        if previo is None or r["timestamp"] >= previo["timestamp"]:
            estado_actual[r["stationId"]] = r.asDict()
    ahora = time.time() * 1000
    L, vacias, llenas = [], 0, 0
    for est, e in estado_actual.items():
        bicis, docks = e["bicisDisponibles"], e["docksDisponibles"]
        ocup = 100.0 * bicis / (bicis + docks) if (bicis + docks) > 0 else None
        vacias += int(bicis == 0 and e["enServicio"])
        llenas += int(docks == 0 and e["enServicio"])
        lab = {"estacion": est, "nombre": e["stationName"] or NOMBRE.get(est, est)}
        L += [metrica("iot_bicis_disponibles", bicis, **lab),
              metrica("iot_ebikes_disponibles", e["ebikesDisponibles"], **lab),
              metrica("iot_docks_disponibles", docks, **lab),
              metrica("iot_ocupacion_pct", ocup, **lab),
              metrica("iot_en_servicio", int(bool(e["enServicio"])), **lab),
              metrica("iot_antiguedad_reporte_s", (ahora - e["timestamp"]) / 1000, **lab)]
    L += [metrica("iot_estaciones_reportando", len(estado_actual)),
          metrica("iot_estaciones_vacias", vacias), metrica("iot_estaciones_llenas", llenas),
          metrica("iot_reportes_nuevos_batch", len(filas))]
    push("iot", L)
    print(f"[iot] batch {batch_id}: {len(filas)} reportes nuevos | vacías={vacias} llenas={llenas}")

estado_iot = (estados.filter("isValid")
    .withWatermark("publishedTime", WATERMARK_IOT)
    .dropDuplicatesWithinWatermark(["stationId", "timestamp"]))

## 6. Consulta `u2_calidad`: contrato, latencia y volumen por topic

Lee **ambos** topics sin filtrar y, por micro-lote, calcula: eventos, válidos, inválidos, latencia de ingesta (`processedAt − publishedAt`, en ms) y bytes por evento. Es la base del panel de observabilidad (criterio 4) y de la estimación de costos (criterio 5).

In [7]:
esquema_min = (StructType().add("tipoEvento", StringType()).add("stationId", StringType())
               .add("timestamp", LongType()).add("publishedAt", LongType())
               .add("bicisDisponibles", IntegerType()).add("docksDisponibles", IntegerType()))

crudo_ambos = leer_topic(f"{TOPIC_RIDES},{TOPIC_ESTACIONES}")
calidad = (crudo_ambos
    .select("topic", F.length("value").alias("bytes"), F.col("value").cast("string").alias("json"))
    .withColumn("e", F.from_json("json", esquema_min))
    .select("topic", "bytes", "e.*")
    .withColumn("isValid", F.coalesce(F.when(F.col("topic") == TOPIC_RIDES, VALIDO_VIAJE).otherwise(VALIDO_ESTADO), F.lit(False)))
    .withColumn("processedAt", F.unix_millis(F.current_timestamp()))
    .withColumn("latencyMs", F.col("processedAt") - F.col("publishedAt")))

totales_calidad = defaultdict(lambda: {"eventos": 0, "validos": 0, "bytes": 0})

def procesar_calidad(df, batch_id):
    resumen = (df.groupBy("topic").agg(
        F.count("*").alias("eventos"), F.sum(F.col("isValid").cast("int")).alias("validos"),
        F.sum("bytes").alias("bytes"), F.avg("latencyMs").alias("lat_avg"),
        F.expr("percentile_approx(latencyMs, 0.95)").alias("lat_p95"),
        F.min("processedAt").alias("t_min"), F.max("processedAt").alias("t_max")).collect())
    if not resumen:
        return
    df.groupBy("topic").agg(F.count("*").alias("eventos"), F.sum(F.col("isValid").cast("int")).alias("validos"),
                            F.sum("bytes").alias("bytes"), F.avg("latencyMs").alias("latenciaPromMs")) \
      .withColumn("batchId", F.lit(batch_id)).withColumn("ts", F.current_timestamp()) \
      .write.mode("append").parquet(f"{STREAM_OUT}/calidad")
    L = []
    for r in resumen:
        t = totales_calidad[r["topic"]]
        t["eventos"] += r["eventos"]; t["validos"] += r["validos"] or 0; t["bytes"] += r["bytes"] or 0
        lab = {"topic": r["topic"]}
        L += [metrica("calidad_eventos_total", t["eventos"], **lab),
              metrica("calidad_invalidos_total", t["eventos"] - t["validos"], **lab),
              metrica("calidad_ratio_validos", t["validos"] / t["eventos"], **lab),
              metrica("calidad_latencia_prom_ms", r["lat_avg"], **lab),
              metrica("calidad_latencia_p95_ms", r["lat_p95"], **lab),
              metrica("calidad_bytes_por_evento", t["bytes"] / t["eventos"], **lab)]
    push("calidad", L)

## 7. Arrancar las consultas y el monitor

`REINICIAR = True` borra checkpoints y salidas para empezar de cero. Como se vio en S08, un checkpoint reutilizado ignora `startingOffsets`. Ponlo en `False` para mostrar en la sustentación que la consulta **retoma desde su checkpoint** después de detenerla.

In [8]:
REINICIAR = True

for q in spark.streams.active:
    q.stop()
if REINICIAR:
    shutil.rmtree(CHECKPOINTS, ignore_errors=True)
    shutil.rmtree(STREAM_OUT, ignore_errors=True)
    iniciar_estado(); estado_actual.clear(); totales_calidad.clear()

q_demanda = (demanda.writeStream.queryName("u2_demanda").outputMode("append")
             .foreachBatch(procesar_ventanas).trigger(processingTime="5 seconds")
             .option("checkpointLocation", f"{CHECKPOINTS}/demanda").start())
q_iot = (estado_iot.writeStream.queryName("u2_estado_iot").outputMode("append")
         .foreachBatch(procesar_estado).trigger(processingTime="10 seconds")
         .option("checkpointLocation", f"{CHECKPOINTS}/estado_iot").start())
q_calidad = (calidad.writeStream.queryName("u2_calidad")
             .foreachBatch(procesar_calidad).trigger(processingTime="5 seconds")
             .option("checkpointLocation", f"{CHECKPOINTS}/calidad").start())
print("Activas:", [q.name for q in spark.streams.active])

Activas: ['u2_demanda', 'u2_estado_iot', 'u2_calidad']


In [9]:
def _num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

def _watermark_ms(p):
    wm = (p.get("eventTime") or {}).get("watermark")
    if not wm or wm.startswith("1970"):
        return None
    return datetime.fromisoformat(wm.replace("Z", "+00:00")).timestamp() * 1000

def monitor(cada_s=10):
    while not parar_monitor.is_set():
        L = []
        for q in spark.streams.active:
            p = q.lastProgress
            if not p:
                continue
            lab = {"query": q.name}
            ops = p.get("stateOperators") or []
            fuente = (p.get("sources") or [{}])[0]
            met = fuente.get("metrics") or {}
            L += [metrica("spark_input_rows_per_s", _num(p.get("inputRowsPerSecond")), **lab),
                  metrica("spark_processed_rows_per_s", _num(p.get("processedRowsPerSecond")), **lab),
                  metrica("spark_num_input_rows", p.get("numInputRows"), **lab),
                  metrica("spark_batch_duration_ms", (p.get("durationMs") or {}).get("triggerExecution"), **lab),
                  metrica("spark_batch_id", p.get("batchId"), **lab),
                  metrica("spark_state_rows", sum(o.get("numRowsTotal", 0) for o in ops), **lab),
                  metrica("spark_rows_dropped_by_watermark", sum(o.get("numRowsDroppedByWatermark", 0) for o in ops), **lab),
                  metrica("spark_duplicados_descartados", sum((o.get("customMetrics") or {}).get("numDroppedDuplicateRows", 0) for o in ops), **lab),
                  metrica("spark_offsets_behind_latest_max", _num(met.get("maxOffsetsBehindLatest")), **lab),
                  metrica("spark_offsets_behind_latest_avg", _num(met.get("avgOffsetsBehindLatest")), **lab),
                  metrica("spark_watermark_ms", _watermark_ms(p), **lab)]
        if L:
            push("monitor", L)
        parar_monitor.wait(cada_s)

parar_monitor = threading.Event()
hilo_monitor = threading.Thread(target=monitor, daemon=True)
hilo_monitor.start()
print("Monitor publicando lastProgress cada 10 s")

Monitor publicando lastProgress cada 10 s


Abre `http://localhost:43000` → **Dashboards** → **U2 · Estaciones — Grimaldo**. Deja las consultas corriendo mientras recorres el tablero y las secciones siguientes. La primera ventana se cierra cuando el watermark pasa su final: unos 25 s reales después de arrancar, con `SPEEDUP=60`.

## 8. Evidencias en vivo (criterios 1-3)

### 8.1 Progreso de cada consulta

In [10]:
time.sleep(30)
for q in spark.streams.active:
    p = q.lastProgress or {}
    print(f"== {q.name}: batch {p.get('batchId')} | filas {p.get('numInputRows')} | "
          f"in {p.get('inputRowsPerSecond')} r/s | proc {p.get('processedRowsPerSecond')} r/s | "
          f"watermark {(p.get('eventTime') or {}).get('watermark')}")
    for o in p.get("stateOperators") or []:
        print(f"   estado: {o.get('operatorName')} filas={o.get('numRowsTotal')} "
              f"descartadas_watermark={o.get('numRowsDroppedByWatermark')} custom={o.get('customMetrics')}")

[demanda] batch 24: 1 ventana(s) cerrada(s), MAE online = 2.699 (120 evaluadas)


[demanda] batch 27: 1 ventana(s) cerrada(s), MAE online = 2.745 (140 evaluadas)


[iot] batch 9: 15 reportes nuevos | vacías=6 llenas=0


[demanda] batch 30: 1 ventana(s) cerrada(s), MAE online = 2.780 (160 evaluadas)


== u2_demanda: batch 30 | filas 1626 | in 325.3 r/s | proc 339.4 r/s | watermark 2026-07-28T13:34:25.435Z
   estado: stateStoreSave filas=3936 descartadas_watermark=0 custom={'SnapshotLastUploaded.partition_0_default': 56, 'SnapshotLastUploaded.partition_1_default': 54, 'SnapshotLastUploaded.partition_2_default': 30, 'SnapshotLastUploaded.partition_3_default': 54, 'forceSnapshotCount': 0, 'loadedMapCacheHitCount': 616, 'loadedMapCacheMissCount': 0, 'numSnapshotsAutoRepaired': 0, 'stateOnCurrentVersionSizeBytes': 1002000}
== u2_estado_iot: batch 9 | filas 20 | in 2.0 r/s | proc 7.7 r/s | watermark 2026-10-30T02:54:09.131Z
   estado: dedupeWithinWatermark filas=104 descartadas_watermark=0 custom={'SnapshotLastUploaded.partition_0_default': -1, 'SnapshotLastUploaded.partition_1_default': -1, 'SnapshotLastUploaded.partition_2_default': -1, 'SnapshotLastUploaded.partition_3_default': -1, 'forceSnapshotCount': 0, 'loadedMapCacheHitCount': 92, 'loadedMapCacheMissCount': 0, 'numDroppedDuplicat

### 8.2 Dato tardío, duplicado e inválido (inyectados a propósito)

Se publican en Kafka, con un `write` batch de Spark (no hace falta otra librería), tres eventos de prueba:

1. **Viaje tardío:** un `viaje.salida` con `timestamp` 2 horas antes del watermark actual. Debe aparecer en `numRowsDroppedByWatermark` de `u2_demanda` y **no** debe cambiar ninguna ventana.
2. **Reporte IoT duplicado:** el mismo `estacion.estado` (misma estación y mismo `timestamp`) enviado 3 veces. `u2_estado_iot` lo cuenta una vez y el resto va a `numDroppedDuplicateRows`.
3. **Evento inválido:** `bicisDisponibles = -3`. Rompe el contrato (rango 0-150) y `u2_calidad` lo suma a inválidos.

In [11]:
def enviar_a_kafka(topic, eventos):
    filas = [(e["stationId"], json.dumps(e)) for e in eventos]
    (spark.createDataFrame(filas, "key string, value string")
     .write.format("kafka").option("kafka.bootstrap.servers", KAFKA).option("topic", topic).save())

wm_ms = _watermark_ms(q_demanda.lastProgress or {}) or (time.time() * 1000)
ahora_ms = int(time.time() * 1000)
est0 = IDS_TOP[0]

tardio = {"tipoEvento": "viaje.salida", "stationId": est0, "stationName": NOMBRE[est0], "rideId": "PRUEBA-TARDIO",
          "rideableType": "classic_bike", "memberCasual": "member", "origen": "prueba-tardio",
          "timestamp": int(wm_ms - 2 * 3600 * 1000), "publishedAt": ahora_ms}
dup = {"tipoEvento": "estacion.estado", "stationId": est0, "gbfsStationId": "prueba", "stationName": NOMBRE[est0],
       "capacidad": 40, "bicisDisponibles": 7, "ebikesDisponibles": 2, "docksDisponibles": 33, "bicisInhabilitadas": 0,
       "docksInhabilitados": 0, "enServicio": True, "origen": "prueba-duplicado",
       "timestamp": ahora_ms - 60_000, "publishedAt": ahora_ms}
invalido = {**dup, "origen": "prueba-invalido", "bicisDisponibles": -3, "timestamp": ahora_ms - 30_000}

caidas_antes = sum(o.get("numRowsDroppedByWatermark", 0) for o in (q_demanda.lastProgress or {}).get("stateOperators", []))
enviar_a_kafka(TOPIC_RIDES, [tardio])
enviar_a_kafka(TOPIC_ESTACIONES, [dup, dup, dup, invalido])
print(f"Inyectados. Watermark de u2_demanda: {datetime.fromtimestamp(wm_ms / 1000, NY):%Y-%m-%d %H:%M:%S} (NYC); "
      f"el viaje tardío tiene {datetime.fromtimestamp(tardio['timestamp'] / 1000, NY):%Y-%m-%d %H:%M:%S}")

Inyectados. Watermark de u2_demanda: 2026-07-28 09:34:25 (NYC); el viaje tardío tiene 2026-07-28 07:34:25


In [12]:
time.sleep(25)
def ultimos(q, n=8):
    return [p for p in q.recentProgress[-n:]]
caidas = sum(o.get("numRowsDroppedByWatermark", 0) for p in ultimos(q_demanda) for o in p.get("stateOperators", []))
dups = sum((o.get("customMetrics") or {}).get("numDroppedDuplicateRows", 0) for p in ultimos(q_iot) for o in p.get("stateOperators", []))
print(f"u2_demanda    -> filas descartadas por watermark en los últimos micro-lotes: {caidas}")
print(f"u2_estado_iot -> duplicados descartados en los últimos micro-lotes: {dups}")
print(f"u2_calidad    -> inválidos acumulados por topic: "
      f"{ {t: v['eventos'] - v['validos'] for t, v in totales_calidad.items()} }")
(spark.read.parquet(f"{STREAM_OUT}/estado_iot").filter(F.col("origen") == "prueba-duplicado")
 .select("stationId", "timestamp", "bicisDisponibles", "batchId").show())

[iot] batch 10: 1 reportes nuevos | vacías=5 llenas=0


[demanda] batch 33: 1 ventana(s) cerrada(s), MAE online = 2.774 (180 evaluadas)


u2_demanda    -> filas descartadas por watermark en los últimos micro-lotes: 1
u2_estado_iot -> duplicados descartados en los últimos micro-lotes: 54
u2_calidad    -> inválidos acumulados por topic: {'citibike-rides': 0, 'citibike-estaciones': 1}


+---------+-------------+----------------+-------+
|stationId|    timestamp|bicisDisponibles|batchId|
+---------+-------------+----------------+-------+
|  6233.04|1793329735323|               7|     10|
+---------+-------------+----------------+-------+



### 8.3 Checkpoint: qué guarda Spark

`offsets/N` guarda el rango de Kafka de cada micro-lote. `commits/N` marca que ese micro-lote terminó. `state/` guarda las ventanas todavía abiertas y las claves de deduplicación.

In [13]:
for nombre in ("demanda", "estado_iot", "calidad"):
    base = f"{CHECKPOINTS}/{nombre}"
    offs = sorted(int(x) for x in os.listdir(f"{base}/offsets") if x.isdigit())
    commits = [x for x in os.listdir(f"{base}/commits") if x.isdigit()] if os.path.isdir(f"{base}/commits") else []
    print(f"== {nombre}: {len(offs)} micro-lotes en offsets/, {len(commits)} en commits/, "
          f"state: {os.path.isdir(f'{base}/state')}")
    if offs:
        print("   último offsets/", offs[-1], "->", open(f"{base}/offsets/{offs[-1]}").read().strip().splitlines()[-1])

== demanda: 37 micro-lotes en offsets/, 36 en commits/, state: True
   último offsets/ 36 -> {"citibike-rides":{"0":25617,"1":26349,"2":29335}}
== estado_iot: 13 micro-lotes en offsets/, 13 en commits/, state: True
   último offsets/ 12 -> {"citibike-estaciones":{"0":70,"1":80,"2":54}}
== calidad: 36 micro-lotes en offsets/, 36 en commits/, state: False
   último offsets/ 35 -> {"citibike-rides":{"0":25617,"1":26349,"2":29335},"citibike-estaciones":{"0":70,"1":80,"2":54}}


## 9. BI y ML en vivo (criterios 6-7)

### 9.1 Ranking de la última ventana cerrada y pronóstico de la siguiente

In [14]:
ventanas = spark.read.parquet(f"{STREAM_OUT}/demanda_ventanas")
ultima = ventanas.agg(F.max("ventanaInicio")).first()[0]
print("Última ventana cerrada (hora NYC):")
(ventanas.filter(F.col("ventanaInicio") == ultima)
 .select(F.date_format("ventanaInicio", "yyyy-MM-dd HH:mm").alias("ventana"), "stationId", "nombre", "salidas", "llegadas")
 .orderBy(F.desc("salidas")).show(10, truncate=False))

pron = sorted(((e, ultima_publicacion.get(("pron", e), 0.0), ultima_publicacion.get(("media_sig", e), 0.0)) for e in IDS_TOP),
              key=lambda x: -x[1])
print("Pronóstico de salidas para la siguiente ventana (top estaciones):")
for e, p, m in pron[:10]:
    marca = "  <- ALERTA (>= 1.5 × media)" if p >= 1.5 * max(m, 1.0) else ""
    print(f"  {e:10s} {NOMBRE[e][:38]:38s} pronóstico={p:6.2f}  media histórica={m:6.2f}{marca}")

[demanda] batch 36: 1 ventana(s) cerrada(s), MAE online = 2.784 (200 evaluadas)


Última ventana cerrada (hora NYC):


+----------------+---------+-----------------------------+-------+--------+
|ventana         |stationId|nombre                       |salidas|llegadas|
+----------------+---------+-----------------------------+-------+--------+
|2026-07-28 09:45|6140.05  |W 21 St & 6 Ave              |14     |13      |
|2026-07-28 09:45|5276.04  |S 2 St & Kent Ave            |12     |15      |
|2026-07-28 09:45|6039.06  |Washington St & Gansevoort St|10     |15      |
|2026-07-28 09:45|6331.01  |W 31 St & 7 Ave              |10     |13      |
|2026-07-28 09:45|5382.06  |Grand St & Elizabeth St      |9      |15      |
|2026-07-28 09:45|5492.05  |Cleveland Pl & Spring St     |9      |8       |
|2026-07-28 09:45|6450.05  |8 Ave & W 31 St              |9      |8       |
|2026-07-28 09:45|6676.02  |W 45 St & 8 Ave              |9      |8       |
|2026-07-28 09:45|5779.11  |Ave A & E 14 St              |9      |8       |
|2026-07-28 09:45|6072.11  |8 Ave & W 16 St              |9      |8       |
+-----------

### 9.2 Validación en producción: predicho vs. real

Cada predicción se compara con lo que realmente pasó cuando esa ventana se cerró. Se compara el MAE en producción con el MAE de test del notebook 01. Si el de producción es mucho peor, hay *drift*: el día reproducido no se parece al período de entrenamiento.

In [15]:
evalua = spark.read.parquet(f"{STREAM_OUT}/evaluacion_online")
k = evalua.agg(F.count("*").alias("n"), F.avg("errorAbs").alias("mae"),
               F.sqrt(F.avg(F.col("errorAbs") ** 2)).alias("rmse")).first()
mae_test = next(r["MAE"] for r in META["resultados"] if r["modelo"] == META["ganador"])
print(f"Predicciones evaluadas: {k['n']} | MAE producción = {k['mae']:.3f} | RMSE producción = {k['rmse']:.3f} | "
      f"MAE test (notebook 01) = {mae_test:.3f}")
(evalua.groupBy("stationId").agg(F.count("*").alias("n"), F.round(F.avg("errorAbs"), 3).alias("MAE"),
                                F.round(F.avg("real"), 2).alias("real_medio"))
 .orderBy(F.desc("MAE")).show(10))

Predicciones evaluadas: 200 | MAE producción = 2.784 | RMSE producción = 3.727 | MAE test (notebook 01) = 1.980


+---------+---+-----+----------+
|stationId|  n|  MAE|real_medio|
+---------+---+-----+----------+
|  5712.03| 10| 4.63|      12.6|
|  6492.08| 10|4.558|       9.8|
|  6233.04| 10|4.064|      14.1|
|  6459.07| 10|3.532|       9.1|
|  6432.09| 10|3.493|      11.9|
|  5905.12| 10|2.965|       7.1|
|  6726.01| 10|2.959|       5.8|
|  6948.10| 10| 2.87|       6.4|
|  6331.01| 10|2.666|       9.1|
|  5763.03| 10| 2.62|       7.7|
+---------+---+-----+----------+
only showing top 10 rows


## 10. Estimación de costos y criterios de escalado (criterio 5)

La estimación sale del **volumen medido** por `u2_calidad`, no de supuestos. Hay que corregir dos sesgos de la demo:
- `citibike-rides` corre **SPEEDUP veces** más rápido que la realidad. La tasa real es la medida dividida entre `SPEEDUP`.
- `citibike-estaciones` solo sigue las top-K. Para todo el sistema se escala a todas las estaciones del feed.

Precios de referencia: los mismos de S9 (AWS us-east-1). **Verifica los precios vigentes en la calculadora de AWS antes de sustentar.**

In [16]:
cal = spark.read.parquet(f"{STREAM_OUT}/calidad")
med = cal.groupBy("topic").agg(F.sum("eventos").alias("eventos"), F.sum("bytes").alias("bytes"),
                               ((F.max(F.unix_timestamp("ts")) - F.min(F.unix_timestamp("ts")))).alias("seg")).collect()
med = {r["topic"]: r for r in med}

ESTACIONES_SISTEMA = 2200          # estaciones activas aprox. en el feed GBFS de NYC (ver log del productor)
INTERVALO_GBFS_S = 30
PRECIO_HORA_T3_MEDIUM = 0.0416     # US$/h (S9)
PRECIO_S3_GB_MES = 0.023           # US$/GB-mes (S9)
HORAS_MES, RETENCION_KAFKA_DIAS = 730, 7

filas = []
for topic, r in med.items():
    if not r["seg"]:
        continue
    bytes_ev = r["bytes"] / r["eventos"]
    tasa_demo = r["eventos"] / r["seg"]
    if topic == TOPIC_RIDES:
        tasa_real = tasa_demo / SPEEDUP
    else:
        tasa_real = ESTACIONES_SISTEMA / INTERVALO_GBFS_S
    gb_dia = tasa_real * 86400 * bytes_ev / 1024**3
    filas.append({"topic": topic, "bytes/evento": round(bytes_ev, 1), "ev/s demo": round(tasa_demo, 2),
                  "ev/s sistema real": round(tasa_real, 2), "GB/día": round(gb_dia, 3),
                  "GB en Kafka (7 d)": round(gb_dia * RETENCION_KAFKA_DIAS, 3), "GB/mes Parquet (crudo)": round(gb_dia * 30, 2)})
import pandas as pd
tabla_costos = pd.DataFrame(filas)
try:
    display(tabla_costos)
except NameError:
    print(tabla_costos.to_string())

gb_mes = tabla_costos["GB/mes Parquet (crudo)"].sum() if len(tabla_costos) else 0
costo_computo = 3 * PRECIO_HORA_T3_MEDIUM * HORAS_MES     # 1 broker Kafka + 1 driver Spark + 1 Prometheus/Grafana
costo_s3 = gb_mes * PRECIO_S3_GB_MES
print(f"\nCómputo: 3 × t3.medium × {HORAS_MES} h = US$ {costo_computo:,.2f}/mes")
print(f"Almacenamiento: {gb_mes:,.2f} GB/mes × US$ {PRECIO_S3_GB_MES} = US$ {costo_s3:,.2f}/mes "
      f"(cota superior: el Parquet comprime el JSON varias veces)")
print(f"Total estimado: US$ {costo_computo + costo_s3:,.2f}/mes")

,topic,bytes/evento,ev/s demo,ev/s sistema real,GB/día,GB en Kafka (7 d),GB/mes Parquet (crudo)
0,citibike-rides,278.0,384.95,6.42,0.143,1.004,4.30
1,citibike-estaciones,383.5,0.75,73.33,2.263,15.842,67.89



Cómputo: 3 × t3.medium × 730 h = US$ 91.10/mes
Almacenamiento: 72.19 GB/mes × US$ 0.023 = US$ 1.66/mes (cota superior: el Parquet comprime el JSON varias veces)
Total estimado: US$ 92.76/mes


### Criterios de escalado

Los umbrales se leen en el tablero, en la fila "Observabilidad":

| Señal (métrica) | Umbral | Acción | Por qué |
|---|---|---|---|
| `spark_batch_duration_ms` de `u2_demanda` | > 5 000 ms sostenido (el trigger) | más núcleos / ejecutores para Spark | si el micro-lote tarda más que el trigger, los lotes se encolan y la latencia crece sin límite |
| `spark_offsets_behind_latest_max` | creciente durante > 2 min | más particiones de `citibike-rides` + más paralelismo de lectura | Spark lee más lento de lo que llega: es *lag* real |
| `spark_state_rows` de `u2_demanda` | crece linealmente sin estabilizarse | revisar el watermark | el estado debe acotarse a ventanas abiertas × estaciones; si no se estabiliza, el watermark no avanza |
| `calidad_latencia_p95_ms` | > 30 000 ms | primero revisar el productor y Kafka, después Spark | la latencia de ingesta incluye la espera en Kafka |
| Estaciones modeladas (K) | de 20 a 200 | mismo modelo global, más filas por micro-lote | el costo de inferencia crece con K (filas), no con la cantidad de modelos |

**Si el volumen se duplica**, primero se escala la **lectura y el agregado en Spark** (más particiones en `citibike-rides` y más paralelismo). El broker tiene holgura: con los valores medidos, el sistema real completo son unos pocos eventos por segundo por topic, muy por debajo de lo que soporta un solo broker.

## 11. Cerrar la sesión

Corre esto al final, después de tomar las capturas del tablero. Con `REINICIAR = False` en la sección 7, las consultas retoman desde su checkpoint.

In [17]:
parar_monitor.set()
for q in spark.streams.active:
    q.stop()
print("Consultas detenidas:", [q.name for q in spark.streams.active] or "ninguna activa")

Consultas detenidas: ninguna activa


Exception in thread "stream execution thread for u2_calidad [id = 2e916d18-1117-4c38-8d50-4f83f44c93e7, runId = a1674c78-830d-4a0a-b46a-21a370c5e09c]" java.lang.StackOverflowError
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.regex.Pattern$CharProperty.match(Pattern.java:4110)
	at java.base/java.util.regex.Pattern$Branch.match(Pattern.java:4914)
	at java.base/java.util.regex.Pattern$GroupHead.match(Pattern.java:4969)
	at java.base/java.util.regex.Pattern$Loop.match(Pattern.java:5078)
	at java.base/java.util.regex.Pattern$GroupTail.match(Pattern.java:5000)
	at java.base/java.util.regex.Pattern$BranchConn.match(Pattern.java:4878)
	at java.base/java.util.rege